# Colab entry point

This is a synthetic setup and checkpoint smoke test; the training loop is a later milestone. Select **Runtime → Change runtime type → GPU** to test CUDA. Python >=3.11 is required. All reusable logic is imported from the editable package in src/.

Run the checkout and installation cells first. If installation requests a runtime restart, restart and rerun the checkout cell, then continue at **Imports** (skip installation).

The repository is private. Opening a notebook through Colab's GitHub integration does not automatically authenticate git clone. Use an existing authenticated checkout, upload the project, or enter a short-lived read-only GitHub token in the hidden prompt. Never save a token in a cell or URL. Set REPO_REF to an immutable commit SHA for a repeatable code checkout; existing checkouts are never silently updated.

In [ ]:
import getpass
import os
import subprocess
import sys
import tempfile
from pathlib import Path

REPO_URL = "https://github.com/BaharAzari/multimodal-agent-rl.git"
REPO_REF = ""  # Optional immutable commit SHA; otherwise record the current HEAD.
candidates = [Path.cwd(), Path.cwd().parent, Path("/content/multimodal-agent-rl")]
PROJECT_ROOT = next(
    (
        path
        for path in candidates
        if (path / "src/multimodal_agent_rl/tools.py").is_file()
    ),
    candidates[-1],
)
if not (PROJECT_ROOT / "src/multimodal_agent_rl/tools.py").is_file():
    token = getpass.getpass("GitHub read token (blank for a public repository): ")
    clone_env = os.environ.copy()
    clone_env["GIT_TERMINAL_PROMPT"] = "0"
    try:
        with tempfile.TemporaryDirectory() as auth_dir:
            if token:
                askpass = Path(auth_dir) / "askpass.py"
                askpass.write_text(
                    "#!/usr/bin/env python3\nimport os, sys\n"
                    "print('x-access-token' if 'username' in sys.argv[1].lower() "
                    "else os.environ['COLAB_GITHUB_TOKEN'])\n"
                )
                askpass.chmod(0o700)
                clone_env.update(GIT_ASKPASS=str(askpass), COLAB_GITHUB_TOKEN=token)
            subprocess.run(
                [
                    "git",
                    "-c",
                    "credential.helper=",
                    "clone",
                    REPO_URL,
                    str(PROJECT_ROOT),
                ],
                env=clone_env,
                check=True,
            )
    finally:
        del token, clone_env
    if REPO_REF:
        subprocess.run(
            ["git", "-C", str(PROJECT_ROOT), "checkout", "--detach", REPO_REF],
            check=True,
        )
os.chdir(PROJECT_ROOT)
if REPO_REF:
    actual = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
    expected = subprocess.check_output(
        ["git", "rev-parse", REPO_REF + "^{commit}"], text=True
    ).strip()
    if actual != expected:
        raise RuntimeError(
            "Existing checkout differs from REPO_REF; use a fresh checkout."
        )
print("Project:", PROJECT_ROOT)
print("Kernel:", sys.executable, sys.version.split()[0])

## Locked installation into the running kernel

The basic workflow is clone → enter the project → editable install. This cell adds exact runtime requirements from uv.lock, including artifact hash checks. It uses the running kernel's Python rather than the local .python-version pin. It does not synchronize or remove Colab's unrelated packages.

Installation can take several minutes because PyTorch includes CUDA libraries. Use a fresh runtime. The final editable install uses the locked setuptools already installed with PyTorch, without fetching separate build dependencies. A restart is required when an already-imported binary package changes. The source checkout and installed files survive a session restart; kernel variables do not.

In [ ]:
import importlib
import site

if sys.version_info < (3, 11):  # noqa: UP036 -- bootstrap runs before package install
    raise RuntimeError("Choose a Colab runtime with Python >=3.11.")
loaded_packages = [name for name in ("torch", "numpy", "PIL") if name in sys.modules]

UV_VERSION = "0.11.13"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", f"uv=={UV_VERSION}"], check=True
)
requirements = (
    Path(tempfile.gettempdir()) / "multimodal-agent-rl-colab-requirements.txt"
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "uv",
        "export",
        "--locked",
        "--no-dev",
        "--no-emit-project",
        "--format",
        "requirements.txt",
        "--python",
        sys.executable,
        "--no-python-downloads",
        "--output-file",
        str(requirements),
    ],
    cwd=PROJECT_ROOT,
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--require-hashes",
        "--force-reinstall",
        "-r",
        str(requirements),
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-deps",
        "--no-build-isolation",
        "-e",
        str(PROJECT_ROOT),
    ],
    check=True,
)
for package_dir in site.getsitepackages():
    site.addsitedir(package_dir)  # Activate the editable src/ path in this live kernel.
importlib.invalidate_caches()
if loaded_packages:
    raise RuntimeError(
        f"Restart the runtime after replacing already-imported {loaded_packages}. "
        "Rerun checkout, skip installation, and continue at Imports."
    )
print("Locked dependencies and editable src/ package installed.")

## Imports

The notebook contains orchestration and examples. Image transformations, device selection, seeding, and checkpoint serialization live in src/.

In [ ]:
import hashlib
import json
import platform
import shutil

import numpy as np
import PIL
import torch
from PIL import Image

import multimodal_agent_rl.tools as project_tools
from multimodal_agent_rl.runtime import (
    load_checkpoint,
    save_checkpoint,
    seed_everything,
    select_device,
)
from multimodal_agent_rl.tools import crop_and_zoom

source_file = Path(project_tools.__file__).resolve()
assert source_file.is_relative_to((PROJECT_ROOT / "src").resolve()), source_file
print("Imported project code:", source_file)

## Device and reproducibility

GPU availability and driver versions vary between Colab sessions. The CUDA probe checks an actual tensor operation; enabling a GPU runtime alone does not guarantee binary compatibility. Set REQUIRE_CUDA=True before future GPU training to fail instead of falling back to CPU. Seeds are reproducible within a compatible environment; they do not guarantee identical results across hardware or nondeterministic kernels.

In [ ]:
SEED = 42
REQUIRE_CUDA = False  # CPU is sufficient for this synthetic smoke test.
device = select_device(require_cuda=REQUIRE_CUDA)
if device.type == "cuda":
    try:
        probe = torch.ones(2, device=device).add_(1).sum().item()
        torch.cuda.synchronize()
        assert probe == 4
    except RuntimeError as error:
        raise RuntimeError(
            "CUDA execution failed. Check the selected runtime/GPU driver "
            "against the locked PyTorch CUDA build before proceeding."
        ) from error
seed_everything(SEED)
git_commit = None
git_dirty = None
if (PROJECT_ROOT / ".git").exists():
    git_commit = subprocess.check_output(
        ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"], text=True
    ).strip()
    git_dirty = bool(
        subprocess.check_output(
            ["git", "-C", str(PROJECT_ROOT), "status", "--porcelain"], text=True
        ).strip()
    )
source_hash = hashlib.sha256()
for path in sorted((PROJECT_ROOT / "src").rglob("*.py")):
    source_hash.update(path.relative_to(PROJECT_ROOT).as_posix().encode())
    source_hash.update(path.read_bytes())
metadata = {
    "seed": SEED,
    "git_commit": git_commit,
    "git_dirty": git_dirty,
    "source_sha256": source_hash.hexdigest(),
    "uv_lock_sha256": hashlib.sha256(
        (PROJECT_ROOT / "uv.lock").read_bytes()
    ).hexdigest(),
    "python": platform.python_version(),
    "torch": str(torch.__version__),
    "numpy": np.__version__,
    "pillow": PIL.__version__,
    "uv_bootstrap": "0.11.13",
    "device": str(device),
    "cuda_runtime": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else None,
}
if shutil.which("nvidia-smi"):
    metadata["nvidia_driver"] = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"],
        capture_output=True,
        text=True,
        check=False,
    ).stdout.strip()
print(json.dumps(metadata, indent=2))

## Checkpoint storage

The default /content/checkpoints is temporary. Set USE_DRIVE=True and grant the Drive mount prompt to persist checkpoints in MyDrive/multimodal-agent-rl/checkpoints. Keep source code and installed packages on the runtime disk. A completed atomic rename protects the previous local file from failed writes, but a Drive mount is not a guarantee that remote synchronization has finished.

In [ ]:
USE_DRIVE = False  # Opt in to persistent storage.
CHECKPOINT_DIR = Path(
    os.environ.get("ANOMALY_RL_CHECKPOINT_DIR", "/content/checkpoints")
)
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    CHECKPOINT_DIR = Path("/content/drive/MyDrive/multimodal-agent-rl/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print("Checkpoint directory:", CHECKPOINT_DIR)

## Synthetic crop/zoom check

In [ ]:
image = Image.new("RGB", (100, 80), "gray")
view = crop_and_zoom(image, (10, 20, 40, 50), zoom_factor=2)
assert view.size == (60, 60)
display(image, view)

## Save and resume a checkpoint

This untrained tiny layer demonstrates serialization only: no optimizer step, dataset, pretrained model, or training loop. The checkpoint contains model/optimizer state dictionaries, step, run metadata, and Python/NumPy/PyTorch RNG states. Restore only your own checkpoints into matching model/optimizer definitions. CPU map_location supports portable loading; CUDA RNG restoration requires a matching GPU count. Future training must also save scheduler, sampler, and data-loader progress.

In [ ]:
model = torch.nn.Linear(2, 1).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
checkpoint_path = save_checkpoint(
    CHECKPOINT_DIR / "smoke.pt",
    model=model,
    optimizer=optimizer,
    step=0,
    metadata=metadata,
)
expected_next_random = torch.rand(3)
restored_model = torch.nn.Linear(2, 1).to(device)
restored_optimizer = torch.optim.AdamW(restored_model.parameters(), lr=1e-3)
payload = load_checkpoint(
    checkpoint_path,
    model=restored_model,
    optimizer=restored_optimizer,
    map_location="cpu",
    restore_rng=True,
)
assert torch.equal(torch.rand(3), expected_next_random)
for key, value in model.state_dict().items():
    assert torch.equal(value, restored_model.state_dict()[key])
assert payload["step"] == 0
assert payload["metadata"]["uv_lock_sha256"] == metadata["uv_lock_sha256"]
print("Saved and restored:", checkpoint_path, "| step:", payload["step"])

## Next milestone

Add the VLM/retrieval/agent and rLLM/GRPO implementation as importable modules under src/. This notebook will call those modules rather than contain a second training implementation. Before real runs, freeze the code revision, data/index versions, configuration, inference budget, and validation protocol described in PROJECT_PLAN.md. Reconnect, repeat the locked setup, mount the same persistent storage, and load a checkpoint to resume.